### ESPCN example

ℹ `xpi-nn-train` is opt for **image classification** benchmarking, here we give a more complex image super-resolution example :)

⚪ Model

In [ ]:
from xpi_nn_train.models.espcn import ESPCN

# for RGB images, upscale x2
model = ESPCN(upscale_factor=2)
model.requires_grad_(True)
print(model)
print('param_cnt:', sum(p.numel() for p in model.parameters() if p.requires_grad))

⚪ Dataset

In [ ]:
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch import Tensor

from xpi_nn_train.datasets.torchvision import get_dataset
from xpi_nn_train.trainers import get_trainer_args

class CIFAR10_SR:

  def __init__(self, split='train'):
    self.dataset = get_dataset('CIFAR10', split)

  def __len__(self) -> int:
    return len(self.dataset)

  def __getitem__(self, index:int) -> tuple[Tensor, Tensor]:
    img, lbl = self.dataset[index]
    img_low = F.interpolate(img.unsqueeze(0), scale_factor=0.5).squeeze(0)
    return (img_low, img)

args = get_trainer_args()
args.epochs = 5
args.learning_rate = 1e-3
print(args)

train_loader = DataLoader(CIFAR10_SR('train'), batch_size=args.batch_size, num_workers=args.num_workers)
test_loader = DataLoader(CIFAR10_SR('val'), batch_size=args.batch_size, num_workers=args.num_workers)
print(f'train: {len(train_loader.dataset)} samples, {len(train_loader)} batches')
print(f'test: {len(test_loader.dataset)} samples, {len(test_loader)} batches')

⚪ Train

Need copy & modify the whole `run_train_native()` training loop :(
Adapt your data batch, loss function and metrics statistics

In [ ]:
from time import time

import torch
import torch.nn.functional as F
from torch.optim import Adam

from xpi_nn_train.utils import seed_everything, ValueWindow

torch.set_num_threads(torch.get_num_interop_threads())
dtype = torch.float32  # fp16/bf16 on most SoC is slow
seed_everything(args.seed)

model = model.to(dtype)
optim = Adam(model.parameters(), lr=args.learning_rate)

vw = ValueWindow(100)
step = 1
ts_init = time()
for epoch in range(1, 1 + args.epochs):
  ts_epoch = time()

  ''' Train '''
  cnt, sum = 0, 0
  model.train()
  optim.zero_grad()
  for x, y in train_loader:
    ts_step = time()
    x = x.to(dtype)
    y = y.to(dtype)
    x_rec = model(x)
    loss = F.mse_loss(x_rec, y)
    if args.grad_acc > 1:
      loss = loss / args.grad_acc
    loss.backward()
    if step % args.grad_acc == 0:
      optim.step()
      optim.zero_grad()
    vw.append(time() - ts_step)

    with torch.inference_mode():
      sum += loss.item()
      cnt += x.size(0)

    step += 1
    if step % vw.nlen == 0:
      print(f'[Step {step}] loss: {loss.item():.5f}, mse: {sum / cnt} ({1 / vw.mean:.5f} it/s)')

  ''' Test '''
  with torch.inference_mode():
    cnt, sum = 0, 0
    model.eval()
    for x, y in test_loader:
      x = x.to(dtype)
      y = y.to(dtype)

      x_rec = model(x)
      loss = F.mse_loss(x_rec, y)
      sum += loss.item()
      cnt += x.size(0)

  print(f'[Epoch {epoch}] Test MSE: {sum / cnt:.3%} ({time() - ts_epoch:.5f}s)\n')

ts_done = time() - ts_init
print(f'Done: {ts_done:.5f}s')